<a href="https://colab.research.google.com/github/Maryam-sudo-png/AI-ML-Internship-Tasks-Phase-2/blob/main/Phase_2_Tasks_Internship.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 1: News Topic Classifier Using BERT**

In [ ]:
pip install transformers datasets torch scikit-learn gradio

In [ ]:
# ==============================
# FIXED TASK 1: BERT CLASSIFIER
# ==============================

from datasets import load_dataset, disable_caching
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import os
import shutil

# Disable caching to force a fresh download and processing
disable_caching()

# Explicitly remove the downloaded 'ag_news' dataset directory if it exists
ag_news_dir = "/content/ag_news"
if os.path.exists(ag_news_dir):
    print(f"Removing existing ag_news dataset directory: {ag_news_dir}")
    shutil.rmtree(ag_news_dir)
    print("Directory removed.")

# 1. Load Dataset
dataset = load_dataset("ag_news", download_mode="force_redownload")

# 🔍 Debug (optional)
print(dataset["train"].column_names)

# 2. Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# ✅ FIX: use correct column name safely
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)

# 3. Rename label column
dataset = dataset.rename_column("label", "labels")

# 4. Set format
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 5. Model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# 6. Metrics
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1": f1_score(p.label_ids, preds, average="weighted")
    }

# 7. Training args
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # ✅ FIX (new versions use eval_strategy)
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,  # keep 1 for fast Colab run
    weight_decay=0.01
)

# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].select(range(2000)),
    eval_dataset=dataset["test"].select(range(500)),
    compute_metrics=compute_metrics
)

# 9. Train
trainer.train()

# 10. Evaluate
print(trainer.evaluate())

# 11. Save
model.save_pretrained("bert-news-model")
tokenizer.save_pretrained("bert-news-model")

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

['text', 'label']


...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.356808,0.890000,0.889063


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.356807678937912, 'eval_accuracy': 0.89, 'eval_f1': 0.8890625481433063, 'eval_runtime': 768.6155, 'eval_samples_per_second': 0.651, 'eval_steps_per_second': 0.082, 'epoch': 1.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert-news-model/tokenizer_config.json', 'bert-news-model/tokenizer.json')

In [ ]:
import gradio as gr
import torch
from transformers import BertTokenizer, BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("bert-news-model")
tokenizer = BertTokenizer.from_pretrained("bert-news-model")

labels = ["World", "Sports", "Business", "Sci/Tech"]

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return labels[pred]

app = gr.Interface(
    fn=predict,
    inputs="text",
    outputs="text",
    title="News Topic Classifier",
)

app.launch()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d0c243e5f4bd55cf6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API**

In [ ]:
pip install pandas scikit-learn joblib


In [ ]:
# Task 2: Churn Prediction Pipeline

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# Load dataset
df = pd.read_csv("Telco-Customer-Churn.csv")

# Preprocessing
df = df.drop("customerID", axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes": 1, "No": 0})

# Column types
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Transformers
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# GridSearch
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10]
}

grid = GridSearchCV(pipeline, param_grid, cv=3)
grid.fit(X_train, y_train)

# Evaluate
y_pred = grid.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# Save model
joblib.dump(grid.best_estimator_, "churn_model.pkl")

print("Model saved successfully!")

Accuracy: 0.7889125799573561
Model saved successfully!


# **Task 5: Auto Tagging Support Tickets Using LLM**

In [ ]:
!pip install transformers torch pandas scikit-learn

In [ ]:
# ==============================
# TASK 5: AUTO TAGGING SUPPORT TICKETS USING LLM
# ==============================
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score
import gradio as gr

data = {
    "ticket": [
        "I can't login to my account",
        "Payment failed but money deducted",
        "App is crashing again and again",
        "Unable to reset my password",
        "Internet connection is very slow",
        "Refund not received yet",
        "My account got hacked",
        "Transaction not showing in history",
        "App freezes when I open it",
        "WiFi disconnects frequently"
    ],
    "true_label": [
        "Login Issue",
        "Payment Issue",
        "Technical Issue",
        "Login Issue",
        "Network Issue",
        "Refund Issue",
        "Security Issue",
        "Payment Issue",
        "Technical Issue",
        "Network Issue"
    ]
}

df = pd.DataFrame(data)

# ------------------------------
# 4. Define Labels
# ------------------------------
labels = [
    "Login Issue",
    "Payment Issue",
    "Technical Issue",
    "Network Issue",
    "Refund Issue",
    "Security Issue"
]

# ------------------------------
# 5. Load Zero-Shot Model
# ------------------------------
print("Loading model... (first time may take time)")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# ------------------------------
# 6. Function: Get Top 3 Tags
# ------------------------------
def get_top_tags(text):
    result = classifier(text, labels)

    # Sort and pick top 3
    top3 = sorted(
        zip(result['labels'], result['scores']),
        key=lambda x: x[1],
        reverse=True
    )[:3]

    return top3

# ------------------------------
# 7. Apply Predictions
# ------------------------------
df["Predicted_Tags"] = df["ticket"].apply(get_top_tags)

# Extract top-1 label for evaluation
df["Predicted_Label"] = df["Predicted_Tags"].apply(lambda x: x[0][0])

# ------------------------------
# 8. Evaluate Model
# ------------------------------
accuracy = accuracy_score(df["true_label"], df["Predicted_Label"])

print("\nModel Accuracy:", accuracy)

# ------------------------------
# 9. Show Results
# ------------------------------
print("\nSample Predictions:\n")
for i in range(len(df)):
    print(f"Ticket: {df['ticket'][i]}")
    print(f"Top 3 Tags: {df['Predicted_Tags'][i]}")
    print(f"Actual: {df['true_label'][i]}")
    print("------")

# ------------------------------
# 10. Few-Shot Prompt Function (Improved)
# ------------------------------
def get_top_tags_few_shot(text):
    prompt = f"""
    Classify the support ticket into relevant categories.

    Examples:
    - "Can't login to account" → Login Issue
    - "Payment failed" → Payment Issue
    - "App crashing" → Technical Issue
    - "Slow internet" → Network Issue
    - "Refund not received" → Refund Issue

    Ticket: {text}
    """

    result = classifier(prompt, labels)

    top3 = sorted(
        zip(result['labels'], result['scores']),
        key=lambda x: x[1],
        reverse=True
    )[:3]

    return top3

# ------------------------------
# 11. Gradio Web App (Deployment)
# ------------------------------
def predict(ticket):
    result = get_top_tags(ticket)

    output = "Top 3 Predictions:\n"
    for tag, score in result:
        output += f"{tag} ({round(score, 2)})\n"

    return output

print("\nLaunching Gradio App...")

app = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=2, placeholder="Enter support ticket here..."),
    outputs="text",
    title="Support Ticket Auto Tagging (LLM)",
    description="Enter a support ticket and get top 3 predicted categories."
)

app.launch()

# ------------------------------
# 12. Save Results (Optional)
# ------------------------------
df.to_csv("ticket_predictions.csv", index=False)

print("\nSaved predictions to ticket_predictions.csv")

Loading model... (first time may take time)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Model Accuracy: 1.0

Sample Predictions:

Ticket: I can't login to my account
Top 3 Tags: [('Login Issue', 0.6171578764915466), ('Security Issue', 0.20317614078521729), ('Technical Issue', 0.09174950420856476)]
Actual: Login Issue
------
Ticket: Payment failed but money deducted
Top 3 Tags: [('Payment Issue', 0.9049311280250549), ('Technical Issue', 0.024156304076313972), ('Login Issue', 0.021758437156677246)]
Actual: Payment Issue
------
Ticket: App is crashing again and again
Top 3 Tags: [('Technical Issue', 0.748525083065033), ('Network Issue', 0.06134706363081932), ('Security Issue', 0.0593411810696125)]
Actual: Technical Issue
------
Ticket: Unable to reset my password
Top 3 Tags: [('Login Issue', 0.41002798080444336), ('Security Issue', 0.39889708161354065), ('Technical Issue', 0.1253446638584137)]
Actual: Login Issue
------
Ticket: Internet connection is very slow
Top 3 Tags: [('Network Issue', 0.4531959891319275), ('Technical Issue', 0.3360118269920349), ('Login Issue', 0.1069


Saved predictions to ticket_predictions.csv
